# Principal Component Analysis

I implemented PCA using feature standardization and covariance eigendecomposition. The notebook checks the implementation against scikit-learn and visualizes one- and two-component projections.

## 🧪 Tests

#### Test Standardization

In [ ]:
%load_ext autoreload
%autoreload 2

from PCA import PCA
import numpy as np

# generate data with zero mean and unit variance
x_train = np.random.randn(3000000, 10)

pca = PCA(new_dim=1)
pca.fit(x_train)

# assert that μ and σ are zero mean and unit variance
assert np.allclose(pca.μ, np.zeros(10), atol=1e-2)
assert np.allclose(pca.σ, np.ones(10), atol=1e-2)

#### Test Fit

In [ ]:
from sklearn.decomposition import PCA as SKPCA
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Generate sample data
x_data= load_iris().data

# Standardize data
x_data = StandardScaler().fit_transform(x_data)

# Define PCA models
pca = PCA(new_dim=2)
sk_pca =  SKPCA(n_components=2)
pca.fit(x_data)
sk_pca.fit(x_data)

assert np.allclose(np.abs(pca.A.T), np.abs(sk_pca.components_.T))

#### Test Transform

In [ ]:
z_data = pca.transform(x_data)
z_data_sk = sk_pca.transform(x_data)
assert np.allclose(np.cov(z_data.T), np.cov(z_data_sk.T))
assert np.allclose(np.cov(z_data.T), np.cov(z_data.T) * np.eye(2))

#### Test Inverse Transform

In [ ]:
# Generate sample data
x_data= load_iris().data

# Define PCA models
pca = PCA(new_dim=4)                # full dimensionality so inverse approximation will be exact
pca.fit(x_data)
z_data = pca.transform(x_data)

assert np.allclose(pca.inverse_transform(z_data), x_data)

## Visualizing PCA Projections

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PCA import PCA  

# Generate data
np.random.seed(40)
μ = [0, 0]  # Mean of the distribution
Σ = [[1, 0.7], [0.7, 1]]  # Covariance matrix

x_data = None
x1, x2 = x_data[:, 0], x_data[:, 1]
y_data = ((x2 - (-1.5 * x1 + -0.1)) > 0) + 1                
corr_line = lambda x: 0.9 * x + 0                   # correlation line

pca = None
z_data = pca.fit_transform(x_data)

pca = None
z_data_1d = pca.fit_transform(x_data)

# Create subplots
plt.style.use('dark_background')
fig, axs = plt.subplots(1, 3, figsize=(18, 6), dpi=200)

# Plot original data and line
axs[0].scatter(x_data[:, 0], x_data[:, 1], s=20, c=y_data, cmap='plasma', edgecolors='white', linewidths=0.3)
axs[0].plot(x_data[:, 0], corr_line(x_data[:, 0]), color='red', linewidth=0.5)
axs[0].set_title("Original Data")

# Plot transformed data by PCA with 2 components
axs[1].scatter(z_data[:, 0], z_data[:, 1], s=20, c=y_data, cmap='plasma', edgecolors='white', linewidths=0.3)
axs[1].set_ylim(-3, 3)
axs[1].set_title("Transformed Data by PCA (2 components)")

# Plot transformed data by PCA with 1 component
axs[2].scatter(z_data_1d, np.zeros_like(z_data_1d), s=20, c=y_data, cmap='plasma', edgecolors='white', linewidths=0.3)
axs[2].set_title("Transformed Data by PCA (1 component)")

plt.tight_layout()
plt.show()

## Comparing Eigendecomposition and SVD

This experiment compares covariance eigenvectors with the right singular vectors of the scaled data matrix.

$$A_{m×n} = U_{m×m} \Sigma_{m×n} V_{n×n}^T$$

It associates the matrix $A$ with:
- $m$ left singular vectors (columns of $U$)
- $n$ right singular vectors (columns of $V$)
- $min(m,n)$ singular values

It can be easily shown that the $n$ right singular vectors (columns of $V$) are the $n$ eigenvectors of $A^TA$ and the $m$ left singular vectors are the $m$ eigenvectors of $AA^T$. 

Meanwhile, the singular values are the square roots of the eigenvalues of either $A^TA$ or $AA^T$


**Consider the following fact:**

In [ ]:
import numpy as np

# Generate random data from the multivariate normal distribution
m = 10000
x_data = np.random.multivariate_normal(μ, Σ, m)

covariance_matrix = x_data.T @ x_data / (m - 1)


assert np.allclose(np.cov(x_data, rowvar=False), covariance_matrix, rtol=0.001)

That is, with zero-meaned (or standardized) data $X_{m×n}$, we have $$Σ=\frac{X^TX}{(m-1)}$$

Thus, we can solve the singular value decomposition of $\frac{X}{\sqrt{m-1}}$ and then use its right singular vectors $A=V^T$ instead of doing eigendecompostion of $Σ$ and then using the eigenvectors $A=U^T$

In [ ]:
# solve eigendecomposition of the covariance matrix
λs, U = np.linalg.eig(np.cov(x_data, rowvar=False))

# solve singular value decomposition of X
U_svd, S, Vt = np.linalg.svd(x_data/(m-1)**0.5)


assert np.allclose(np.abs(U), np.abs(Vt.T), rtol=1e-2)
assert (np.allclose(np.flip(λs), S**2, rtol=1e-2) or np.allclose(λs, S**2, rtol=1e-2))                    